In [1]:
# Importation de la fonction load_dataset depuis Hugging Face Datasets
from datasets import load_dataset

# Chargement d’un dataset local au format CSV (fichier .txt structuré en CSV)
# column_names définit les noms des colonnes : texte + label
dataset = load_dataset(
    "csv",
    data_files="../data/sample_texts.txt",
    column_names=["text", "label"]
)

# Affichage des 5 premiers exemples du dataset
print(dataset["train"][:5])

FileNotFoundError: Unable to find 'C:/Users/adilu/Desktop/GI-S4/DEEP/DeepLearning_Project/TP4_Transformers/notebooks\../data/sample_texts.txt'

In [ ]:
# Importation du tokenizer automatique de Hugging Face
from transformers import AutoTokenizer

# Définition du modèle pré-entraîné utilisé (DistilBERT)
model_name = "distilbert-base-uncased"

# Chargement du tokenizer associé au modèle
# Le tokenizer transforme le texte en tokens numériques
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Fonction de tokenisation appliquée à chaque exemple du dataset
def tokenize(example):
    return tokenizer(
        example["text"],        # colonne texte du dataset
        padding="max_length",   # padding pour avoir une longueur fixe
        truncation=True,        # coupe les textes trop longs
        max_length=32           # taille maximale des séquences
    )

# Application de la tokenisation sur tout le dataset
# batched=True permet de traiter plusieurs exemples en même temps (plus rapide)
dataset = dataset.map(tokenize, batched=True)

In [ ]:
# Renommage de la colonne "label" en "labels"
# (format attendu par les modèles Hugging Face)
dataset = dataset.rename_column("label", "labels")

# Conversion du dataset en format PyTorch
# On sélectionne uniquement les colonnes nécessaires pour l'entraînement
dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# Affichage du premier exemple après transformation
print(dataset["train"][0])

In [ ]:
# Importation du modèle de classification de séquences
from transformers import AutoModelForSequenceClassification

# Chargement du modèle pré-entraîné (DistilBERT) adapté à la classification
# num_labels=2 car on a une classification binaire (positif / négatif)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

In [ ]:
# Importation des arguments d'entraînement de Hugging Face
from transformers import TrainingArguments

# Définition des paramètres d'entraînement du modèle
training_args = TrainingArguments(
    output_dir="./results",          # dossier de sortie des résultats
    num_train_epochs=3,              # nombre d'époques d'entraînement
    per_device_train_batch_size=2,   # taille du batch par GPU/CPU
    learning_rate=5e-5,              # taux d'apprentissage
    logging_dir="./logs",            # dossier pour les logs
    logging_steps=1,                 # fréquence d'affichage des logs
    report_to="none"                 # désactive les outils de tracking externes
)

In [ ]:
# Importation du Trainer de Hugging Face
from transformers import Trainer

# Création du Trainer pour gérer l'entraînement du modèle
trainer = Trainer(
    model=model,                     # modèle à entraîner
    args=training_args,              # paramètres d'entraînement
    train_dataset=dataset["train"]   # dataset d'entraînement
)

In [ ]:
# Lancement de l'entraînement du modèle
# Le Trainer gère automatiquement la boucle d'entraînement
trainer.train()